# Aprendizaje no supervisado: K-Means, DBSCAN y detección de anomalías

## Caso de estudio: Corporación Favorita

En este laboratorio se utilizará un dataset semanal de aproximadamente **23 millones de registros**.

El flujo será:

1. procesamiento con Polars;
2. construcción de un perfil analítico por producto;
3. exploración de las variables;
4. estandarización;
5. segmentación con K-Means;
6. diagnóstico de una limitación de K-Means;
7. clustering basado en densidad con DBSCAN;
8. detección de productos anómalos;
9. comparación e interpretación de negocio.


## 1. Instalación de bibliotecas

Google Colab ya incluye varias de las bibliotecas necesarias. Se instala Polars para trabajar eficientemente con el archivo de gran tamaño.

In [1]:
!pip install -q polars pyarrow plotly scikit-learn

## 2. Importación de bibliotecas

Se utilizarán:

- **Polars** para procesar los 23 millones de registros.
- **Pandas** para trabajar con el dataset reducido.
- **Scikit-learn** para el escalamiento y K-Means.
- **Plotly** para todos los gráficos interactivos.

In [2]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import polars as pl

import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", None)

tiempos = []
inicio_total = perf_counter()

## 3. Función para registrar tiempos

La función siguiente guarda y muestra el tiempo utilizado por cada proceso relevante.

In [3]:
def registrar_tiempo(proceso, inicio):
    segundos = perf_counter() - inicio
    tiempos.append({
        "Proceso": proceso,
        "Tiempo_segundos": segundos
    })
    print(f"{proceso}: {segundos:.2f} segundos")
    return segundos

## 4. Carga de `Favorita_weekly.parquet`

El archivo debe estar disponible en:

`/content/Favorita_weekly.parquet`

La carga se realiza con Polars debido al volumen del dataset.

In [5]:
RUTA_ARCHIVO = Path("/content/favorita_weekly.parquet")

if not RUTA_ARCHIVO.exists():
    raise FileNotFoundError(
        "No se encontró /content/Favorita_weekly.parquet. "
        "Sube el archivo a Google Colab y vuelve a ejecutar la celda."
    )

inicio = perf_counter()

df = pl.read_parquet(RUTA_ARCHIVO)

tiempo_carga = registrar_tiempo("Carga del archivo con Polars", inicio)

print(f"Filas: {df.height:,}")
print(f"Columnas: {df.width}")

Carga del archivo con Polars: 0.62 segundos
Filas: 23,582,329
Columnas: 8


## 5. Revisión inicial

Antes de transformar los datos, revisamos las columnas, sus tipos y los primeros registros.

In [6]:
print("Columnas:")
print(df.columns)

print("\nTipos de datos:")
print(df.schema)

df.head()

Columnas:
['fecha_semana', 'anio', 'semana', 'store_nbr', 'item_nbr', 'unit_sales', 'dias_promocion', 'dias_con_venta']

Tipos de datos:
Schema({'fecha_semana': Datetime(time_unit='ns', time_zone=None), 'anio': Int16, 'semana': Int8, 'store_nbr': Int16, 'item_nbr': Int32, 'unit_sales': Float32, 'dias_promocion': Int8, 'dias_con_venta': Int8})


fecha_semana,anio,semana,store_nbr,item_nbr,unit_sales,dias_promocion,dias_con_venta
datetime[ns],i16,i8,i16,i32,f32,i8,i8
2012-12-31 00:00:00,2013,1,1,103520,5.0,0,2
2012-12-31 00:00:00,2013,1,1,103665,13.0,0,5
2012-12-31 00:00:00,2013,1,1,105574,26.0,0,5
2012-12-31 00:00:00,2013,1,1,105575,38.0,0,5
2012-12-31 00:00:00,2013,1,1,105577,11.0,0,5


### Interpretación

El dataset semanal mantiene una granularidad operacional. Un producto aparece muchas veces porque puede estar presente en distintas tiendas y semanas.

Por ello, **no corresponde aplicar K-Means directamente sobre estas filas**.

## 6. Gráfico del volumen del dataset

El gráfico permite dimensionar visualmente el número de registros y columnas.

In [7]:
volumen = pd.DataFrame({
    "Medida": ["Filas", "Columnas"],
    "Cantidad": [df.height, df.width]
})

fig = px.bar(
    volumen,
    x="Medida",
    y="Cantidad",
    text_auto=",",
    title="Dimensión del dataset original"
)

fig.show()

## 7. Limpieza simple

Para mantener el foco en aprendizaje no supervisado, se aplicarán únicamente dos operaciones:

- eliminar registros con valores nulos;
- eliminar registros duplicados.

In [8]:
registros_originales = df.height

inicio = perf_counter()
df = df.drop_nulls()
tiempo_nulos = registrar_tiempo("Eliminación de valores nulos", inicio)
registros_sin_nulos = df.height

inicio = perf_counter()
df = df.unique()
tiempo_duplicados = registrar_tiempo("Eliminación de duplicados", inicio)
registros_limpios = df.height

print(f"Registros originales:       {registros_originales:,}")
print(f"Después de eliminar nulos:  {registros_sin_nulos:,}")
print(f"Después de duplicados:      {registros_limpios:,}")

Eliminación de valores nulos: 0.06 segundos
Eliminación de duplicados: 10.67 segundos
Registros originales:       23,582,329
Después de eliminar nulos:  23,582,329
Después de duplicados:      23,582,329


## 8. Efecto de la limpieza

In [9]:
control_limpieza = pd.DataFrame({
    "Etapa": [
        "Dataset original",
        "Sin valores nulos",
        "Sin duplicados"
    ],
    "Registros": [
        registros_originales,
        registros_sin_nulos,
        registros_limpios
    ]
})

fig = px.bar(
    control_limpieza,
    x="Etapa",
    y="Registros",
    text_auto=",",
    title="Cantidad de registros durante la limpieza"
)

fig.show()

## 9. Evolución semanal de la demanda

Antes de construir los segmentos, observamos cómo evoluciona la demanda total a través del tiempo.

In [10]:
inicio = perf_counter()

demanda_semanal = (
    df
    .group_by("fecha_semana")
    .agg(
        pl.col("unit_sales").sum().alias("demanda_total")
    )
    .sort("fecha_semana")
    .to_pandas()
)

tiempo_demanda_semanal = registrar_tiempo(
    "Construcción del resumen semanal",
    inicio
)

fig = px.line(
    demanda_semanal,
    x="fecha_semana",
    y="demanda_total",
    title="Evolución semanal de la demanda total"
)

fig.show()

Construcción del resumen semanal: 0.38 segundos


### Interpretación

Este gráfico permite identificar cambios, aumentos o caídas en la cantidad vendida a lo largo del tiempo. Su objetivo es contextualizar el comportamiento general antes de segmentar los productos.

## 10. Cantidad de productos activos por semana

In [11]:
productos_semana = (
    df
    .group_by("fecha_semana")
    .agg(
        pl.col("item_nbr").n_unique().alias("productos_activos")
    )
    .sort("fecha_semana")
    .to_pandas()
)

fig = px.line(
    productos_semana,
    x="fecha_semana",
    y="productos_activos",
    title="Cantidad de productos activos por semana"
)

fig.show()

## 11. Construcción del perfil de cada producto

K-Means requiere una matriz en la cual:

- cada fila represente una observación;
- cada columna represente una característica.

En este problema, cada fila debe representar un **producto**.

Para cada producto se calcularán las siguientes variables:

- demanda total;
- demanda promedio;
- variabilidad de la demanda;
- demanda máxima;
- cantidad de semanas activas;
- cantidad de tiendas activas;
- promedio de días con venta;
- promedio de días en promoción.

In [12]:
inicio = perf_counter()

perfil_productos_pl = (
    df
    .group_by("item_nbr")
    .agg(
        pl.col("unit_sales").sum().alias("demanda_total"),
        pl.col("unit_sales").mean().alias("demanda_promedio"),
        pl.col("unit_sales").std().fill_null(0).alias("variabilidad_demanda"),
        pl.col("unit_sales").max().alias("demanda_maxima"),
        pl.col("fecha_semana").n_unique().alias("semanas_activas"),
        pl.col("store_nbr").n_unique().alias("tiendas_activas"),
        pl.col("dias_con_venta").mean().alias("promedio_dias_con_venta"),
        pl.col("dias_promocion").mean().alias("promedio_dias_promocion")
    )
    .sort("item_nbr")
)

tiempo_perfil = registrar_tiempo(
    "Construcción del perfil de productos",
    inicio
)

print(f"Productos: {perfil_productos_pl.height:,}")
print(f"Variables: {perfil_productos_pl.width}")
perfil_productos_pl.head()

Construcción del perfil de productos: 4.38 segundos
Productos: 4,036
Variables: 9


item_nbr,demanda_total,demanda_promedio,variabilidad_demanda,demanda_maxima,semanas_activas,tiendas_activas,promedio_dias_con_venta,promedio_dias_promocion
i32,f32,f32,f32,f32,u32,u32,f64,f64
96995,10143.0,5.987603,17.108437,650.0,106,31,3.086777,0.0
99197,17870.0,13.86346,25.239382,404.0,88,28,3.802948,0.000776
103501,164753.0,29.031366,16.037376,114.0,238,30,6.315595,0.149075
103520,201117.0,18.363495,19.935404,380.0,242,54,4.855278,0.164354
103665,219669.0,25.433483,15.430075,149.0,242,41,5.841033,0.267338


## 12. Comparación entre el dataset operacional y el analítico

La transformación reduce aproximadamente 23 millones de registros a una fila por producto.

In [13]:
comparacion_volumen = pd.DataFrame({
    "Dataset": [
        "Dataset semanal",
        "Perfil de productos"
    ],
    "Filas": [
        registros_limpios,
        perfil_productos_pl.height
    ]
})

fig = px.bar(
    comparacion_volumen,
    x="Dataset",
    y="Filas",
    text_auto=",",
    log_y=True,
    title="Reducción del dataset operacional al dataset analítico"
)

fig.show()

### Interpretación

La escala logarítmica permite visualizar una diferencia muy grande entre ambos tamaños.

El dataset original sirve para almacenar el comportamiento semanal. El perfil reducido es el dataset apropiado para aplicar clustering.

## 13. Conversión del perfil a Pandas

Una vez reducido el volumen, Pandas es suficiente para continuar con el análisis y utilizar Scikit-learn.

In [14]:
inicio = perf_counter()

productos = perfil_productos_pl.to_pandas()

tiempo_conversion = registrar_tiempo(
    "Conversión de Polars a Pandas",
    inicio
)

productos.head()

Conversión de Polars a Pandas: 0.01 segundos


,item_nbr,demanda_total,demanda_promedio,variabilidad_demanda,demanda_maxima,semanas_activas,tiendas_activas,promedio_dias_con_venta,promedio_dias_promocion
0,96995,10143.0,5.987603,17.108437,650.0,106,31,3.086777,0.000000
1,99197,17870.0,13.863460,25.239382,404.0,88,28,3.802948,0.000776
2,103501,164753.0,29.031366,16.037376,114.0,238,30,6.315595,0.149075
3,103520,201117.0,18.363495,19.935404,380.0,242,54,4.855278,0.164354
4,103665,219669.0,25.433483,15.430075,149.0,242,41,5.841033,0.267338


## 14. Resumen estadístico del perfil

In [15]:
productos.describe().T

,count,mean,std,min,25%,50%,75%,max
item_nbr,4036.0,1.238921e+06,582513.498462,96995.0,812007.500000,1.261200e+06,1.696041e+06,2.127114e+06
demanda_total,4036.0,2.660085e+05,410154.968750,1.0,56584.750000,1.572115e+05,3.006348e+05,6.264200e+06
demanda_promedio,4036.0,4.645903e+01,127.744316,1.0,15.238127,2.780577e+01,4.977235e+01,7.024801e+03
variabilidad_demanda,4036.0,5.327573e+01,289.369354,0.0,14.568553,2.744344e+01,5.173481e+01,1.769072e+04
demanda_maxima,4036.0,8.410552e+02,2340.535400,1.0,154.000000,3.240000e+02,7.323132e+02,8.944000e+04
semanas_activas,4036.0,1.686231e+02,72.798692,1.0,111.750000,1.710000e+02,2.420000e+02,2.420000e+02
tiendas_activas,4036.0,4.328171e+01,12.776146,1.0,29.000000,5.200000e+01,5.400000e+01,5.400000e+01
promedio_dias_con_venta,4036.0,5.009193e+00,1.291949,1.0,4.377805,5.373270e+00,5.953374e+00,6.932357e+00
promedio_dias_promocion,4036.0,3.711456e-01,0.414896,0.0,0.043804,2.267070e-01,6.071911e-01,3.965217e+00


## 15. Distribución de las principales variables

Las variables de demanda presentan una fuerte asimetría. Para visualizarlas sin perder productos con valor cero, aplicaremos:

```python
np.log1p(x)
```

Esta transformación equivale a calcular `ln(1 + x)`.


In [16]:
productos["log_demanda_promedio"] = np.log1p(
    productos["demanda_promedio"]
)

fig = px.histogram(
    productos,
    x="log_demanda_promedio",
    nbins=50,
    title="Distribución de la demanda promedio por producto"
)

fig.update_layout(
    xaxis_title="ln(1 + demanda promedio)",
    yaxis_title="Cantidad de productos"
)

fig.show()


## 16. Distribución de la demanda total


In [17]:
productos["log_demanda_total"] = np.log1p(
    productos["demanda_total"]
)

fig = px.histogram(
    productos,
    x="log_demanda_total",
    nbins=50,
    title="Distribución de la demanda total por producto"
)

fig.update_layout(
    xaxis_title="ln(1 + demanda total)",
    yaxis_title="Cantidad de productos"
)

fig.show()


## 17. Distribución de la variabilidad de la demanda


In [18]:
productos["log_variabilidad_demanda"] = np.log1p(
    productos["variabilidad_demanda"]
)

fig = px.histogram(
    productos,
    x="log_variabilidad_demanda",
    nbins=50,
    title="Distribución de la variabilidad de la demanda"
)

fig.update_layout(
    xaxis_title="ln(1 + variabilidad)",
    yaxis_title="Cantidad de productos"
)

fig.show()


## 18. Relación entre demanda promedio y variabilidad

Este gráfico permite observar:

- la relación entre nivel de demanda y variabilidad;
- la concentración principal de productos;
- la presencia de productos extremadamente diferentes.

Los valores se muestran en escala logarítmica para evitar que unos pocos productos compriman toda la nube.


In [19]:
fig = px.scatter(
    productos,
    x="demanda_promedio",
    y="variabilidad_demanda",
    hover_data=[
        "item_nbr",
        "demanda_total",
        "semanas_activas",
        "tiendas_activas"
    ],
    log_x=True,
    log_y=True,
    opacity=0.60,
    title="Demanda promedio y variabilidad por producto"
)

fig.show()


### Interpretación

Se observa una relación positiva entre la demanda promedio y la variabilidad de la demanda: los productos que venden más tienden también a presentar mayores fluctuaciones en sus ventas. La mayoría de los productos se concentra en una región central, mientras que algunos casos aparecen claramente alejados del resto, lo que podría indicar productos de comportamiento excepcional o posibles anomalías. Debido a la gran diferencia de magnitudes entre las variables, se utiliza una escala logarítmica para facilitar la visualización y, posteriormente, será necesario estandarizar los datos antes de aplicar algoritmos de clustering como K-Means.

## 19. Cobertura y demanda promedio

Como `tiendas_activas` es una variable discreta, utilizaremos un boxplot en lugar de una nube de puntos.


In [20]:
fig = px.box(
    productos,
    x="tiendas_activas",
    y="demanda_promedio",
    points="outliers",
    log_y=True,
    title="Demanda promedio según cantidad de tiendas activas"
)

fig.update_layout(
    xaxis_title="Cantidad de tiendas activas",
    yaxis_title="Demanda promedio"
)

fig.show()


### Interpretación

El boxplot muestra cómo varía la demanda promedio según la cantidad de tiendas donde está disponible cada producto. Se observa una alta dispersión en los productos con menor cobertura, mientras que aquellos presentes en un mayor número de tiendas tienden a presentar una demanda más estable. También existen valores atípicos, correspondientes a productos con un comportamiento excepcional que podrían analizarse posteriormente mediante técnicas de detección de anomalías.

## 21. Selección de variables para K-Means

Se seleccionan variables que describen diferentes dimensiones del comportamiento de cada producto:

- nivel de demanda;
- variabilidad;
- cobertura;
- continuidad;
- promoción.

In [21]:
variables = [
    "demanda_promedio",
    "variabilidad_demanda",
    "demanda_maxima",
    "semanas_activas",
    "tiendas_activas",
    "promedio_dias_promocion"
]

X = productos[variables].copy()

X.head()

,demanda_promedio,variabilidad_demanda,demanda_maxima,semanas_activas,tiendas_activas,promedio_dias_promocion
0,5.987603,17.108437,650.0,106,31,0.000000
1,13.863460,25.239382,404.0,88,28,0.000776
2,29.031366,16.037376,114.0,238,30,0.149075
3,18.363495,19.935404,380.0,242,54,0.164354
4,25.433483,15.430075,149.0,242,41,0.267338


## 20. Matriz de correlaciones

La matriz permite observar qué variables contienen información similar y cuáles aportan dimensiones diferentes.

**Lectura esperada**

- Las variables asociadas a demanda pueden presentar correlaciones elevadas.
- Cobertura, continuidad y promociones pueden aportar información complementaria.
- Más adelante utilizaremos PCA únicamente para visualizar el espacio multivariable.


In [22]:
correlaciones = X.corr()

fig = px.imshow(
    correlaciones,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    title="Correlación entre las variables del perfil"
)

fig.show()


### Interpretación

La matriz de correlación muestra que **demanda promedio**, **variabilidad de la demanda** y **demanda máxima** presentan una alta correlación positiva, indicando que describen aspectos relacionados del comportamiento de ventas. En cambio, variables como **semanas activas**, **tiendas activas** y **promedio de días en promoción** muestran correlaciones bajas, aportando información complementaria. Esto justifica el uso de técnicas de clustering, que consideran simultáneamente todas las variables para identificar patrones que no son evidentes al analizar una sola característica.

## 23. Escalas originales

K-Means calcula distancias. Por esta razón, las variables con valores numéricos más altos podrían dominar el resultado si no se estandarizan.

In [23]:
datos_largos = X.melt(
    var_name="Variable",
    value_name="Valor"
)

fig = px.box(
    datos_largos,
    x="Variable",
    y="Valor",
    points=False,
    title="Variables antes de la estandarización"
)

fig.show()

## 24. Estandarización de variables

In [24]:
inicio = perf_counter()

escalador = StandardScaler()
X_escalado = escalador.fit_transform(X)

tiempo_escalamiento = registrar_tiempo(
    "Estandarización de variables",
    inicio
)

X_escalado_df = pd.DataFrame(
    X_escalado,
    columns=variables
)

X_escalado_df.head()

Estandarización de variables: 0.01 segundos


,demanda_promedio,variabilidad_demanda,demanda_maxima,semanas_activas,tiendas_activas,promedio_dias_promocion
0,-0.316855,-0.125002,-0.081639,-0.860330,-0.961420,-0.894662
1,-0.255194,-0.096900,-0.186756,-1.107618,-1.196261,-0.892792
2,-0.136443,-0.128704,-0.310675,0.953114,-1.039700,-0.535311
3,-0.219963,-0.115231,-0.197012,1.008067,0.839033,-0.498481
4,-0.164611,-0.130803,-0.295719,1.008067,-0.178614,-0.250232


## 25. Variables después de la estandarización

In [25]:
escalados_largos = X_escalado_df.melt(
    var_name="Variable",
    value_name="Valor estandarizado"
)

fig = px.box(
    escalados_largos,
    x="Variable",
    y="Valor estandarizado",
    points=False,
    title="Variables después de la estandarización"
)

fig.show()

### Interpretación

Tras la estandarización, todas las variables quedan en una escala comparable, con media cercana a cero y desviación estándar unitaria. Esto evita que variables con valores más grandes dominen el cálculo de las distancias en algoritmos como K-Means. Los valores extremos que aún se observan corresponden a productos con comportamientos excepcionales (outliers), los cuales serán analizados posteriormente mediante técnicas de detección de anomalías como DBSCAN.

## 25. Evaluación de distintos valores de K

Utilizaremos tres criterios:

1. Método del Codo.
2. Silhouette Score.
3. Interpretación de negocio.

En datos reales los indicadores no siempre coinciden.


In [26]:
K_FINAL = 4

inicio = perf_counter()

resultados = []

for k in range(2, 11):
    modelo = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    etiquetas = modelo.fit_predict(X_escalado)

    resultados.append({
        "k": k,
        "inercia": modelo.inertia_,
        "silhouette": silhouette_score(X_escalado, etiquetas)
    })

resultados_k = pd.DataFrame(resultados)

tiempo_evaluacion_k = registrar_tiempo(
    "Evaluación de valores de K",
    inicio
)

resultados_k


Evaluación de valores de K: 2.84 segundos


,k,inercia,silhouette
0,2,16064.105172,0.972411
1,3,12550.871777,0.274062
2,4,10252.953005,0.295207
3,5,8610.695179,0.306917
4,6,7177.046214,0.309590
5,7,6104.522878,0.336842
6,8,5610.351209,0.346936
7,9,5098.590621,0.350328
8,10,4642.993225,0.352673


## 26. Método del Codo

La curva no siempre presenta un punto de inflexión evidente. Por eso el Método del Codo funciona como una herramienta de apoyo, no como una regla automática.


In [27]:
fig = px.line(
    resultados_k,
    x="k",
    y="inercia",
    markers=True,
    title="Método del Codo"
)

fig.add_vline(
    x=K_FINAL,
    line_dash="dash",
    annotation_text=f"K seleccionado = {K_FINAL}"
)

fig.update_xaxes(dtick=1)
fig.show()


### Interpretación

El Método del Codo permite estimar un número adecuado de clústeres observando cómo disminuye la inercia a medida que aumenta el valor de **K**. En este caso, la reducción de la inercia es muy pronunciada entre **K = 2** y **K = 4**; a partir de ese punto, las mejoras son cada vez menores. Por ello, se selecciona **K = 4** como un buen equilibrio entre representar adecuadamente la estructura de los datos y evitar una segmentación excesivamente compleja. No obstante, esta decisión se complementará con el **Silhouette Score** y la interpretación de negocio.

## 27. Silhouette Score

Un valor alto indica clústeres compactos y separados. Sin embargo, el máximo estadístico no siempre entrega la segmentación más útil para el negocio.

La decisión final combinará los indicadores con la interpretabilidad de los segmentos.


In [28]:
fig = px.line(
    resultados_k,
    x="k",
    y="silhouette",
    markers=True,
    title="Silhouette Score según número de clústeres"
)

silhouette_final = resultados_k.loc[
    resultados_k["k"] == K_FINAL,
    "silhouette"
].iloc[0]

fig.add_vline(
    x=K_FINAL,
    line_dash="dash",
    annotation_text=f"K seleccionado = {K_FINAL}"
)

fig.update_xaxes(dtick=1)
fig.show()


### Interpretación

El **Silhouette Score** mide qué tan compactos y bien separados están los clústeres. En este caso, el valor más alto se obtiene con **K = 2**, lo que indica una separación muy marcada entre dos grandes grupos. Sin embargo, esta solución resulta demasiado simple para el objetivo del análisis, ya que no captura la diversidad de comportamientos presentes en los productos. Por ello, se selecciona **K = 4**, logrando un equilibrio entre la calidad estadística de la segmentación y una interpretación más útil desde el punto de vista del negocio.

## 28. Decisión final sobre K

En este caso:

- el Codo no es completamente evidente;
- Silhouette favorece una solución muy simple;
- el negocio necesita una segmentación más rica e interpretable.

Por eso utilizaremos `K_FINAL = 4` y posteriormente verificaremos si todos los clústeres representan segmentos reales.


In [29]:
print("Método del Codo: no presenta un punto completamente definido")
print("Silhouette: favorece una solución más simple")
print("Criterio de negocio: buscamos segmentos interpretables")
print("K seleccionado:", K_FINAL)


Método del Codo: no presenta un punto completamente definido
Silhouette: favorece una solución más simple
Criterio de negocio: buscamos segmentos interpretables
K seleccionado: 4


## 30. Entrenamiento del modelo final

In [30]:
inicio = perf_counter()

modelo_final = KMeans(
    n_clusters=K_FINAL,
    random_state=42,
    n_init=10
)

productos["cluster"] = modelo_final.fit_predict(X_escalado)
productos["cluster"] = productos["cluster"].astype(str)

tiempo_kmeans = registrar_tiempo(
    "Entrenamiento del modelo K-Means final",
    inicio
)

productos.head()

Entrenamiento del modelo K-Means final: 0.04 segundos


,item_nbr,demanda_total,demanda_promedio,variabilidad_demanda,demanda_maxima,semanas_activas,tiendas_activas,promedio_dias_con_venta,promedio_dias_promocion,log_demanda_promedio,log_demanda_total,log_variabilidad_demanda,cluster
0,96995,10143.0,5.987603,17.108437,650.0,106,31,3.086777,0.000000,1.944138,9.224638,2.896378,0
1,99197,17870.0,13.863460,25.239382,404.0,88,28,3.802948,0.000776,2.698906,9.790935,3.267262,0
2,103501,164753.0,29.031366,16.037376,114.0,238,30,6.315595,0.149075,3.402242,12.012209,2.835410,0
3,103520,201117.0,18.363495,19.935404,380.0,242,54,4.855278,0.164354,2.963390,12.211647,3.041442,2
4,103665,219669.0,25.433483,15.430075,149.0,242,41,5.841033,0.267338,3.274632,12.299882,2.799114,2


## 30. Tamaño y diagnóstico de los clústeres

Además de entrenar K-Means, debemos verificar si los grupos obtenidos tienen tamaños razonables.

Un clúster extremadamente pequeño puede representar:

- un segmento de nicho;
- una observación excepcional;
- o una limitación del algoritmo.


In [31]:
tamano_clusters = (
    productos["cluster"]
    .value_counts()
    .rename_axis("cluster")
    .reset_index(name="cantidad_productos")
    .sort_values("cluster")
)

tamano_clusters["porcentaje"] = (
    tamano_clusters["cantidad_productos"]
    / len(productos)
    * 100
)

tamano_clusters["etiqueta"] = (
    tamano_clusters["cantidad_productos"].map("{:,}".format)
    + " ("
    + tamano_clusters["porcentaje"].map("{:.2f}%".format)
    + ")"
)

fig = px.bar(
    tamano_clusters,
    x="cluster",
    y="cantidad_productos",
    text="etiqueta",
    title="Cantidad y porcentaje de productos por clúster"
)

fig.show()

clusters_pequenos = tamano_clusters[
    tamano_clusters["porcentaje"] < 1
]

if len(clusters_pequenos) > 0:
    print("ADVERTENCIA: existen clústeres con menos del 1% de los productos.")
    display(clusters_pequenos)


ADVERTENCIA: existen clústeres con menos del 1% de los productos.


,cluster,cantidad_productos,porcentaje,etiqueta
3,1,1,0.024777,1 (0.02%)


### Interpretación

La distribución de los productos muestra que la mayoría se concentra en tres clústeres de tamaño considerable, mientras que el **clúster 1 contiene un solo producto (0,02%)**. Esto sugiere que K-Means ha aislado un caso extremadamente diferente del resto, lo que podría corresponder a un producto atípico más que a un verdadero segmento de negocio. Este resultado motiva el uso de **DBSCAN**, un algoritmo capaz de identificar automáticamente observaciones anómalas sin obligarlas a formar un clúster independiente.

## 32. Distribución de la demanda promedio por clúster

El boxplot permite comparar la mediana, la dispersión y los valores extremos de la demanda promedio.

In [32]:
fig = px.box(
    productos,
    x="cluster",
    y="demanda_promedio",
    color="cluster",
    points="outliers",
    log_y=True,
    title="Distribución de la demanda promedio por clúster"
)

fig.show()

## 33. Distribución de la cobertura por clúster

La cantidad de tiendas activas permite interpretar el alcance comercial de cada segmento.

In [33]:
fig = px.box(
    productos,
    x="cluster",
    y="tiendas_activas",
    color="cluster",
    points="outliers",
    title="Distribución de la cobertura por clúster"
)

fig.show()

## 34. Distribución de la variabilidad por clúster

Este gráfico permite comparar qué grupos contienen productos más estables y cuáles presentan mayor incertidumbre.

In [34]:
fig = px.box(
    productos,
    x="cluster",
    y="variabilidad_demanda",
    color="cluster",
    points="outliers",
    log_y=True,
    title="Distribución de la variabilidad de la demanda por clúster"
)

fig.show()

## 34. Perfil promedio de cada clúster

Primero conservamos una tabla con los promedios originales para facilitar la interpretación de negocio.


In [35]:
perfil_clusters = (
    productos
    .groupby("cluster")[variables]
    .mean()
    .round(2)
)

perfil_clusters

,demanda_promedio,variabilidad_demanda,demanda_maxima,semanas_activas,tiendas_activas,promedio_dias_promocion
cluster,,,,,,
0,39.259998,35.160000,463.140015,166.36,27.04,0.26
1,7024.799805,17690.720703,89440.000000,24.00,1.00,0.00
2,38.590000,44.020000,1003.820007,212.15,51.77,0.22
3,61.860001,73.309998,902.330017,94.57,47.02,0.76


## 35. Mapa de calor del perfil estandarizado

Este es el gráfico principal para interpretar los clústeres.

- Valores positivos: sobre el promedio general.
- Valores negativos: bajo el promedio general.
- Valores cercanos a cero: comportamiento promedio.

Los clústeres con muy pocos productos deben interpretarse con precaución.


In [36]:
perfil_escalado = X_escalado_df.copy()
perfil_escalado["cluster"] = productos["cluster"].values

perfil_escalado = (
    perfil_escalado
    .groupby("cluster")
    .mean()
)

fig = px.imshow(
    perfil_escalado,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    title="Perfil estandarizado de cada clúster"
)

fig.show()

print(
    "Nota: un clúster con muy pocos productos no representa "
    "necesariamente un segmento de negocio."
)


Nota: un clúster con muy pocos productos no representa necesariamente un segmento de negocio.


### Interpretación

El mapa de calor muestra el comportamiento promedio de cada clúster respecto del promedio general de los datos estandarizados. Los valores positivos indican características sobre el promedio, mientras que los negativos representan valores inferiores al promedio. Se observa que los clústeres **0, 2 y 3** presentan perfiles relativamente equilibrados, mientras que el **clúster 1** exhibe valores extremadamente altos en demanda y variabilidad. Dado que este clúster está formado por un solo producto, es probable que corresponda a una observación atípica más que a un verdadero segmento de negocio, lo que justifica analizar posteriormente los datos mediante DBSCAN.

## 36. Clústeres según demanda y variabilidad

La visualización permite observar la nube principal y detectar productos extremadamente alejados.

El punto aislado que K-Means convierte en un clúster será revisado posteriormente mediante DBSCAN.


In [37]:
fig = px.scatter(
    productos,
    x="demanda_promedio",
    y="variabilidad_demanda",
    color="cluster",
    hover_data=[
        "item_nbr",
        "demanda_total",
        "demanda_maxima",
        "semanas_activas",
        "tiendas_activas"
    ],
    log_x=True,
    log_y=True,
    opacity=0.55,
    title="Clústeres K-Means según demanda y variabilidad"
)

fig.show()


### Interpretación

La proyección de los productos según su demanda promedio y variabilidad muestra que los clústeres obtenidos por K-Means presentan un cierto grado de superposición, lo que indica que la separación entre grupos no es completamente evidente cuando se observan solo estas dos variables. Sin embargo, se identifica un producto claramente aislado del resto, que K-Means ha asignado a un clúster independiente. Este resultado sugiere la presencia de una posible anomalía y motiva la aplicación de DBSCAN para determinar si corresponde a un verdadero segmento o a un comportamiento atípico.

## 37. Visualización mediante PCA

**Importante:** PCA se utiliza únicamente para proyectar los datos en dos dimensiones.

K-Means fue entrenado con todas las variables estandarizadas, no solamente con las dos componentes mostradas.


In [38]:
inicio = perf_counter()

pca = PCA(n_components=2)
componentes_pca = pca.fit_transform(X_escalado)

productos["PCA_1"] = componentes_pca[:, 0]
productos["PCA_2"] = componentes_pca[:, 1]

tiempo_pca = registrar_tiempo(
    "Proyección de los clústeres con PCA",
    inicio
)

varianza_pca = pca.explained_variance_ratio_.sum()

print(
    f"Varianza explicada por las dos componentes: "
    f"{varianza_pca:.2%}"
)

Proyección de los clústeres con PCA: 0.03 segundos
Varianza explicada por las dos componentes: 64.03%


## 38. Clústeres K-Means proyectados en dos componentes

El gráfico permite observar:

- la concentración principal de productos;
- la separación parcial de los segmentos;
- la presencia de productos muy alejados del comportamiento predominante.


In [39]:
fig = px.scatter(
    productos,
    x="PCA_1",
    y="PCA_2",
    color="cluster",
    hover_data=[
        "item_nbr",
        "demanda_promedio",
        "variabilidad_demanda",
        "tiendas_activas",
        "semanas_activas"
    ],
    opacity=0.55,
    title="PCA de los clústeres obtenidos con K-Means"
)

fig.show()


### Interpretación

La proyección de los productos según su demanda promedio y variabilidad muestra que los clústeres obtenidos por K-Means presentan un cierto grado de superposición, lo que indica que la separación entre grupos no es completamente evidente cuando se observan solo estas dos variables. Sin embargo, se identifica un producto claramente aislado del resto, que K-Means ha asignado a un clúster independiente. Este resultado sugiere la presencia de una posible anomalía y motiva la aplicación de DBSCAN para determinar si corresponde a un verdadero segmento o a un comportamiento atípico.

## 39. Resumen ejecutivo de K-Means

La siguiente tabla resume tamaño, demanda, variabilidad, cobertura y promociones de cada clúster.


In [40]:
resumen_ejecutivo = (
    productos
    .groupby("cluster")
    .agg(
        cantidad_productos=("item_nbr", "count"),
        demanda_promedio=("demanda_promedio", "mean"),
        variabilidad_promedio=("variabilidad_demanda", "mean"),
        semanas_activas_promedio=("semanas_activas", "mean"),
        tiendas_activas_promedio=("tiendas_activas", "mean"),
        promocion_promedio=("promedio_dias_promocion", "mean")
    )
    .round(2)
    .reset_index()
)

resumen_ejecutivo

,cluster,cantidad_productos,demanda_promedio,variabilidad_promedio,semanas_activas_promedio,tiendas_activas_promedio,promocion_promedio
0,0,1185,39.259998,35.160000,166.36,27.04,0.26
1,1,1,7024.799805,17690.720703,24.00,1.00,0.00
2,2,1819,38.590000,44.020000,212.15,51.77,0.22
3,3,1031,61.860001,73.309998,94.57,47.02,0.76


## 46. Resumen de tiempos de ejecución

In [41]:
tiempo_total = perf_counter() - inicio_total

tiempos.append({
    "Proceso": "Tiempo total del laboratorio",
    "Tiempo_segundos": tiempo_total
})

tiempos_df = pd.DataFrame(tiempos)

tiempos_df

,Proceso,Tiempo_segundos
0,Carga del archivo con Polars,0.617088
1,Eliminación de valores nulos,0.060327
2,Eliminación de duplicados,10.673930
3,Construcción del resumen semanal,0.377864
4,Construcción del perfil de productos,4.377450
5,Conversión de Polars a Pandas,0.005455
6,Estandarización de variables,0.008044
7,Evaluación de valores de K,2.844158
8,Entrenamiento del modelo K-Means final,0.043128
9,Proyección de los clústeres con PCA,0.029194


## 47. Gráfico de tiempos de ejecución

In [42]:
tiempos_procesos = tiempos_df[
    tiempos_df["Proceso"] != "Tiempo total del laboratorio"
].copy()

fig = px.bar(
    tiempos_procesos,
    x="Proceso",
    y="Tiempo_segundos",
    text_auto=".2f",
    title="Tiempo de ejecución por proceso"
)

fig.show()

print(f"Tiempo total del laboratorio: {tiempo_total:.2f} segundos")

Tiempo total del laboratorio: 816.16 segundos


# Cierre de K-Means: una pregunta importante

K-Means:

- segmentó todos los productos;
- necesitó definir previamente el número de clústeres;
- obligó a todas las observaciones a pertenecer a un grupo;
- generó al menos un clúster extremadamente pequeño.

## Pregunta

> ¿Ese clúster representa realmente un segmento o corresponde a un producto anómalo?

Para responder utilizaremos **DBSCAN**, que puede identificar observaciones que no pertenecen a ninguna región densa.


# Parte II. DBSCAN y detección de anomalías

En esta segunda parte reutilizaremos exactamente el mismo dataset analítico.

No repetiremos:

- carga de datos;
- limpieza;
- agregación;
- conversión a Pandas;
- selección de variables;
- estandarización;
- cálculo de PCA.

Trabajaremos directamente con:

```python
X_escalado
```

Esto permite comparar K-Means y DBSCAN utilizando las mismas observaciones y variables.


## 49. Importación de DBSCAN y herramientas auxiliares

DBSCAN utiliza dos parámetros principales:

- `eps`: distancia máxima para considerar que dos observaciones son vecinas;
- `min_samples`: cantidad mínima de observaciones necesarias para formar una región densa.

También utilizaremos `NearestNeighbors` para construir un gráfico que ayude a seleccionar un valor razonable de `eps`.


In [43]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors


## 50. Función para resumir un resultado DBSCAN

DBSCAN asigna:

- etiquetas `0, 1, 2, ...` a los clústeres encontrados;
- etiqueta `-1` a las observaciones clasificadas como ruido.

En este laboratorio interpretaremos las observaciones con etiqueta `-1` como **posibles anomalías**.


In [44]:
def resumir_dbscan(etiquetas):
    etiquetas = np.asarray(etiquetas)

    n_clusters = len(set(etiquetas)) - (1 if -1 in etiquetas else 0)
    n_anomalias = int((etiquetas == -1).sum())
    porcentaje_anomalias = n_anomalias / len(etiquetas) * 100

    return {
        "clusters": n_clusters,
        "anomalias": n_anomalias,
        "porcentaje_anomalias": porcentaje_anomalias
    }


## 42. Selección de `eps` mediante la distancia al k-ésimo vecino

Buscamos el punto donde las distancias comienzan a crecer rápidamente.

- Antes del cambio: productos en regiones densas.
- Después del cambio: productos potencialmente aislados.

Usaremos `min_samples = 10`.


In [45]:
MIN_SAMPLES_INICIAL = 10

inicio = perf_counter()

vecinos = NearestNeighbors(
    n_neighbors=MIN_SAMPLES_INICIAL
)

vecinos.fit(X_escalado)

distancias, _ = vecinos.kneighbors(X_escalado)

distancias_k = np.sort(
    distancias[:, -1]
)

tiempo_vecinos = registrar_tiempo(
    "Cálculo de distancias para DBSCAN",
    inicio
)

distancias_k[:10]


Cálculo de distancias para DBSCAN: 0.13 segundos


array([0.02383039, 0.0259219 , 0.02638561, 0.02690798, 0.02690798,
       0.02803294, 0.02803294, 0.0293092 , 0.02945191, 0.03029709])

In [46]:
EPS_FINAL = 1.2
MIN_SAMPLES_FINAL = 10

distancias_df = pd.DataFrame({
    "Producto_ordenado": np.arange(1, len(distancias_k) + 1),
    "Distancia": distancias_k
})

fig = px.line(
    distancias_df,
    x="Producto_ordenado",
    y="Distancia",
    title=(
        "Selección de eps mediante la distancia "
        f"al {MIN_SAMPLES_FINAL}.º vecino"
    )
)

fig.add_hline(
    y=EPS_FINAL,
    line_dash="dash",
    annotation_text=f"eps seleccionado = {EPS_FINAL}"
)

fig.update_layout(
    xaxis_title="Productos ordenados por distancia",
    yaxis_title="Distancia"
)

fig.show()


### Interpretación

El gráfico muestra la distancia de cada producto a su **10.º vecino más cercano**, ordenadas de menor a mayor. La mayoría de las observaciones presenta distancias pequeñas, lo que indica que pertenecen a regiones densas del espacio de datos. Hacia el final de la curva se observa un incremento brusco de la distancia, correspondiente a productos más aislados. El valor de **`eps = 1.2`** se selecciona antes de ese cambio, permitiendo que DBSCAN forme clústeres en las zonas densas e identifique automáticamente como anomalías las observaciones que se encuentran demasiado alejadas del resto.

## 43. Parámetros seleccionados

Los valores pueden ajustarse durante la clase, pero utilizaremos una configuración consistente con el gráfico anterior.


In [47]:
print("eps final:", EPS_FINAL)
print("min_samples final:", MIN_SAMPLES_FINAL)


eps final: 1.2
min_samples final: 10


## 44. Entrenamiento de DBSCAN

DBSCAN no necesita que indiquemos previamente el número de clústeres.

Las observaciones que no pertenecen a ninguna región densa reciben la etiqueta `-1`.


In [48]:
inicio = perf_counter()

modelo_dbscan_final = DBSCAN(
    eps=EPS_FINAL,
    min_samples=MIN_SAMPLES_FINAL
)

productos["cluster_dbscan"] = modelo_dbscan_final.fit_predict(
    X_escalado
)

tiempo_dbscan_final = registrar_tiempo(
    "Entrenamiento DBSCAN final",
    inicio
)

resumen_dbscan_final = resumir_dbscan(
    productos["cluster_dbscan"]
)

resumen_dbscan_final


Entrenamiento DBSCAN final: 0.33 segundos


{'clusters': 1, 'anomalias': 97, 'porcentaje_anomalias': 2.4033696729435086}

## 45. Distribución de resultados DBSCAN

DBSCAN puede encontrar:

- clústeres densos;
- productos clasificados como anomalías (`-1`).

Mostraremos cantidad y porcentaje.


In [49]:
productos["segmento_dbscan"] = np.where(
    productos["cluster_dbscan"] == -1,
    "Anomalías (-1)",
    "Clúster " + productos["cluster_dbscan"].astype(str)
)

distribucion_dbscan = (
    productos["segmento_dbscan"]
    .value_counts()
    .rename_axis("segmento_dbscan")
    .reset_index(name="cantidad_productos")
)

distribucion_dbscan["porcentaje"] = (
    distribucion_dbscan["cantidad_productos"]
    / len(productos)
    * 100
)

distribucion_dbscan["etiqueta"] = (
    distribucion_dbscan["cantidad_productos"].map("{:,}".format)
    + " ("
    + distribucion_dbscan["porcentaje"].map("{:.2f}%".format)
    + ")"
)

distribucion_dbscan


,segmento_dbscan,cantidad_productos,porcentaje,etiqueta
0,Clúster 0,3939,97.59663,"3,939 (97.60%)"
1,Anomalías (-1),97,2.40337,97 (2.40%)


In [50]:
fig = px.bar(
    distribucion_dbscan,
    x="segmento_dbscan",
    y="cantidad_productos",
    color="segmento_dbscan",
    text="etiqueta",
    title="Distribución de productos según DBSCAN"
)

fig.update_layout(
    xaxis_title="Resultado DBSCAN",
    yaxis_title="Cantidad de productos",
    showlegend=False
)

fig.show()


### Interpretación

DBSCAN identificó un único clúster principal que agrupa al **97,6% de los productos**, mientras que **97 productos (2,4%)** fueron clasificados como anomalías (`-1`). A diferencia de K-Means, DBSCAN no obliga a que todas las observaciones pertenezcan a un clúster, permitiendo separar automáticamente los productos con comportamientos significativamente diferentes. Estos casos atípicos representan oportunidades para identificar productos estrella, promociones excepcionales, errores en los datos o comportamientos poco frecuentes que requieren un análisis específico.

## 46. Visualización de DBSCAN mediante el mismo PCA

Se reutilizan exactamente las mismas componentes calculadas para K-Means.

Así, la comparación visual entre ambos algoritmos es directa.


In [51]:
fig = px.scatter(
    productos,
    x="PCA_1",
    y="PCA_2",
    color="segmento_dbscan",
    hover_data=[
        "item_nbr",
        "demanda_promedio",
        "variabilidad_demanda",
        "tiendas_activas",
        "semanas_activas"
    ],
    title="PCA de los resultados obtenidos con DBSCAN"
)

fig.update_traces(
    marker={"size": 7, "opacity": 0.55}
)

fig.show()


### Interpretación

La proyección mediante PCA permite visualizar los resultados de DBSCAN en dos dimensiones. Se observa que la gran mayoría de los productos forma una región densa (Clúster 0), mientras que las observaciones clasificadas como **anomalías** aparecen dispersas y alejadas de esta concentración principal. Esto confirma que DBSCAN no crea clústeres artificiales para los casos extremos, sino que identifica automáticamente aquellos productos cuyo comportamiento difiere significativamente del resto.

## 47. Productos detectados como anomalías

Una anomalía no es necesariamente un error.

Puede corresponder a:

- un producto estrella;
- una promoción extraordinaria;
- baja cobertura;
- un comportamiento excepcional;
- o un problema de calidad de datos.


In [52]:
anomalias = (
    productos[
        productos["cluster_dbscan"] == -1
    ]
    .copy()
)

print(f"Cantidad de anomalías: {len(anomalias):,}")
print(
    "Porcentaje del total:",
    f"{len(anomalias) / len(productos):.2%}"
)

anomalias.head()


Cantidad de anomalías: 97
Porcentaje del total: 2.40%


,item_nbr,demanda_total,demanda_promedio,variabilidad_demanda,demanda_maxima,semanas_activas,tiendas_activas,promedio_dias_con_venta,promedio_dias_promocion,log_demanda_promedio,log_demanda_total,log_variabilidad_demanda,cluster,PCA_1,PCA_2,cluster_dbscan,segmento_dbscan
5,105574,522163.0,86.637299,165.441650,11316.0,242,29,6.690227,0.485316,4.473207,13.165737,5.114645,0,2.809567,0.570487,-1,Anomalías (-1)
92,161288,197748.0,33.630611,355.124451,13833.0,160,53,4.288095,0.548299,3.544738,12.194754,5.875280,2,3.533228,0.644625,-1,Anomalías (-1)
218,261052,2403946.0,199.414856,209.741196,7122.0,242,54,6.783409,0.951058,5.300389,14.692622,5.350631,2,2.541215,0.589306,-1,Anomalías (-1)
224,264299,1045671.0,88.212502,377.987366,12537.0,242,54,6.237894,0.417412,4.491021,13.860170,5.937503,2,3.502270,1.575914,-1,Anomalías (-1)
240,265559,4112827.0,341.143585,389.319366,26955.0,242,54,6.888437,0.222628,5.835230,15.229622,5.966965,2,7.930273,2.498931,-1,Anomalías (-1)


## 48. Productos anómalos de mayor demanda identificados por DBSCAN

Estos son los casos prioritarios para una revisión comercial y de calidad de datos.


In [53]:
top_anomalias_demanda = (
    anomalias
    .nlargest(20, "demanda_total")
)

fig = px.bar(
    top_anomalias_demanda,
    x="item_nbr",
    y="demanda_total",
    hover_data=[
        "demanda_promedio",
        "variabilidad_demanda",
        "tiendas_activas",
        "semanas_activas"
    ],
    title="Productos anómalos de mayor demanda identificados por DBSCAN"
)

fig.update_layout(
    xaxis_title="Código del producto (item_nbr)",
    yaxis_title="Demanda total"
)

fig.update_xaxes(type="category")
fig.update_yaxes(type="log")
fig.show()


### Interpretación

El gráfico muestra los productos que DBSCAN identificó como anomalías y que presentan la mayor demanda total. Estos productos poseen un comportamiento significativamente diferente al resto del catálogo, por lo que fueron clasificados como observaciones atípicas en lugar de incorporarse al clúster principal. En un contexto de negocio, estos casos deben analizarse individualmente, ya que pueden corresponder a productos estrella, promociones de gran impacto, artículos estacionales o incluso posibles inconsistencias en los datos.

## 49. ¿Qué caracteriza a las anomalías?

Compararemos productos normales y anomalías utilizando las variables estandarizadas.

Esto evita mezclar magnitudes incompatibles y permite identificar qué características se encuentran sobre o bajo el promedio.


In [54]:
productos["tipo_observacion"] = np.where(
    productos["cluster_dbscan"] == -1,
    "Anomalías",
    "Productos normales"
)

perfil_anomalias = X_escalado_df.copy()
perfil_anomalias["tipo_observacion"] = productos["tipo_observacion"].values

perfil_anomalias = (
    perfil_anomalias
    .groupby("tipo_observacion")
    .mean()
)

perfil_anomalias


,demanda_promedio,variabilidad_demanda,demanda_maxima,semanas_activas,tiendas_activas,promedio_dias_promocion
tipo_observacion,,,,,,
Anomalías,2.317217,1.789858,3.284581,-0.472403,-0.362614,1.193372
Productos normales,-0.057063,-0.044076,-0.080885,0.011633,0.008930,-0.029387


In [55]:
fig = px.imshow(
    perfil_anomalias,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    title="Perfil estandarizado: productos normales vs anomalías"
)

fig.show()


### Interpretación

El mapa de calor compara el perfil promedio de los **productos normales** y las **anomalías** identificadas por DBSCAN. Se observa que las anomalías presentan valores muy superiores en **demanda promedio**, **demanda máxima** y **variabilidad de la demanda**, mientras que tienden a estar presentes en menos semanas y en un menor número de tiendas. Además, muestran un mayor promedio de días en promoción. Estas diferencias confirman que las anomalías corresponden a productos con un comportamiento significativamente distinto al resto del catálogo.

## 50. K-Means vs DBSCAN

| Aspecto | K-Means | DBSCAN |
|---|---|---|
| Criterio | Distancia a centroides | Densidad |
| Número de grupos | Se define previamente | Surge de los datos |
| Asigna todos los productos | Sí | No |
| Detecta anomalías | No | Sí |
| Parámetros principales | `K` | `eps`, `min_samples` |
| Uso en este laboratorio | Segmentación | Clustering y anomalías |


## 51. ¿Qué ocurrió con los clústeres de K-Means al aplicar DBSCAN?

El heatmap permite observar qué productos permanecen en la región principal y cuáles son reclasificados como anomalías.


In [57]:
comparacion_modelos = (
    productos
    .groupby(
        ["cluster", "segmento_dbscan"]
    )
    .size()
    .reset_index(name="cantidad_productos")
)

comparacion_modelos


,cluster,segmento_dbscan,cantidad_productos
0,0,Anomalías (-1),18
1,0,Clúster 0,1167
2,1,Anomalías (-1),1
3,2,Anomalías (-1),29
4,2,Clúster 0,1790
5,3,Anomalías (-1),49
6,3,Clúster 0,982


In [58]:
fig = px.density_heatmap(
    comparacion_modelos,
    x="cluster",
    y="segmento_dbscan",
    z="cantidad_productos",
    text_auto=True,
    color_continuous_scale="Blues",
    title="¿Qué ocurrió con los clústeres K-Means al aplicar DBSCAN?"
)

fig.update_xaxes(title="Clúster K-Means")
fig.update_yaxes(title="Resultado DBSCAN")
fig.show()

print(
    "Observe especialmente el clúster pequeño de K-Means: "
    "DBSCAN puede reclasificarlo como anomalía."
)


Observe especialmente el clúster pequeño de K-Means: DBSCAN puede reclasificarlo como anomalía.


### Interpretación

El mapa de calor compara la asignación de productos realizada por **K-Means** y **DBSCAN**. Se observa que la mayoría de los productos permanece en el clúster principal identificado por DBSCAN, mientras que un pequeño conjunto de observaciones es reclasificado como **anomalías**. En particular, el clúster muy pequeño generado por K-Means no se mantiene como un segmento independiente, sino que sus productos son tratados como casos atípicos. Esto demuestra que DBSCAN distingue entre grupos naturales y observaciones aisladas, evitando la creación de clústeres artificiales.

## Conclusiones

En este laboratorio se compararon dos algoritmos de aprendizaje no supervisado aplicados a un conjunto de productos de retail.

- **K-Means** permitió segmentar los productos en grupos con características similares, siendo una herramienta útil para tareas de clasificación y segmentación comercial.
- **DBSCAN** identificó automáticamente un conjunto reducido de productos con comportamientos atípicos, sin necesidad de definir previamente el número de clústeres.
- La comparación entre ambos métodos muestra que la elección del algoritmo depende del objetivo del análisis: segmentar la población o detectar anomalías.

En aplicaciones reales, ambos enfoques son complementarios y pueden utilizarse conjuntamente para comprender mejor el comportamiento de los datos y apoyar la toma de decisiones.

## 66. Productos que requieren revisión de negocio

Exportaremos una tabla con las anomalías detectadas para facilitar su revisión posterior.


In [59]:
columnas_revision = [
    "item_nbr",
    "demanda_total",
    "demanda_promedio",
    "variabilidad_demanda",
    "demanda_maxima",
    "semanas_activas",
    "tiendas_activas",
    "promedio_dias_con_venta",
    "promedio_dias_promocion",
    "cluster",
    "cluster_dbscan"
]

anomalias_revision = (
    anomalias[columnas_revision]
    .sort_values(
        "demanda_total",
        ascending=False
    )
)

anomalias_revision.head(20)


,item_nbr,demanda_total,demanda_promedio,variabilidad_demanda,demanda_maxima,semanas_activas,tiendas_activas,promedio_dias_con_venta,promedio_dias_promocion,cluster,cluster_dbscan
2809,1503844,6264200.00,1481.949341,1296.176514,6361.227051,150,43,6.501774,0.907736,3,-1
1451,1047679,5512875.00,457.196472,729.642883,6451.000000,242,54,6.693730,0.573478,2,-1
2739,1473474,4990322.50,702.961365,773.342712,4906.736816,150,53,6.766728,0.943091,3,-1
393,364606,4416426.00,366.234833,237.774200,5938.000000,242,54,6.908367,0.097769,2,-1
1033,819932,4410943.00,382.761444,368.863617,5809.000000,239,54,6.520653,0.121225,2,-1
2601,1463992,4408545.00,570.907166,666.177979,3877.000000,152,54,6.435250,0.047268,2,-1
1004,807493,4321424.00,362.870422,369.851990,4519.000000,242,54,6.707952,0.052565,2,-1
240,265559,4112827.00,341.143585,389.319366,26955.000000,242,54,6.888437,0.222628,2,-1
658,559870,3532720.00,293.123138,360.450958,22231.000000,242,54,6.846416,0.614089,2,-1
722,584028,3252203.00,532.798645,476.656372,2985.294922,242,39,6.833552,0.837156,3,-1


In [60]:
ruta_anomalias = "productos_anomalos_dbscan.csv"

anomalias_revision.to_csv(
    ruta_anomalias,
    index=False
)

print(
    f"Archivo generado: {ruta_anomalias}"
)


Archivo generado: productos_anomalos_dbscan.csv


## 67. Actualización de los tiempos de ejecución

Agregamos al resumen los procesos asociados a DBSCAN.


In [61]:
tiempo_total_actualizado = perf_counter() - inicio_total

tiempos_actualizados = pd.DataFrame(tiempos).copy()

fila_total = tiempos_actualizados[
    tiempos_actualizados["Proceso"] == "Tiempo total del laboratorio"
].index

if len(fila_total) > 0:
    tiempos_actualizados.loc[
        fila_total,
        "Tiempo_segundos"
    ] = tiempo_total_actualizado
else:
    tiempos_actualizados = pd.concat(
        [
            tiempos_actualizados,
            pd.DataFrame([{
                "Proceso": "Tiempo total del laboratorio",
                "Tiempo_segundos": tiempo_total_actualizado
            }])
        ],
        ignore_index=True
    )

tiempos_actualizados


,Proceso,Tiempo_segundos
0,Carga del archivo con Polars,0.617088
1,Eliminación de valores nulos,0.060327
2,Eliminación de duplicados,10.673930
3,Construcción del resumen semanal,0.377864
4,Construcción del perfil de productos,4.377450
5,Conversión de Polars a Pandas,0.005455
6,Estandarización de variables,0.008044
7,Evaluación de valores de K,2.844158
8,Entrenamiento del modelo K-Means final,0.043128
9,Proyección de los clústeres con PCA,0.029194


In [62]:
tiempos_sin_total = tiempos_actualizados[
    tiempos_actualizados["Proceso"] != "Tiempo total del laboratorio"
].copy()

fig = px.bar(
    tiempos_sin_total,
    x="Proceso",
    y="Tiempo_segundos",
    text_auto=".2f",
    title="Tiempos de ejecución del laboratorio completo"
)

fig.update_xaxes(tickangle=-45)
fig.show()


# 54. Conclusiones finales

## K-Means

- Segmentó todos los productos.
- Requirió seleccionar previamente `K`.
- Permitió construir perfiles de negocio.
- Generó un clúster extremadamente pequeño, lo que motivó un diagnóstico adicional.

## DBSCAN

- Descubrió regiones densas sin definir previamente el número de grupos.
- No obligó a todos los productos a pertenecer a un clúster.
- Identificó automáticamente productos anómalos.
- Generó una lista priorizada para revisión.

## Conclusión de negocio

Los productos anómalos no deben eliminarse automáticamente.

Pueden representar:

- productos estrella;
- promociones excepcionales;
- errores de datos;
- eventos poco frecuentes;
- oportunidades comerciales.

El valor del análisis está en reducir miles de productos a un conjunto pequeño de casos que el negocio puede investigar.
